# 04 — Consumer LangGraph

Build the consumer state machine, render its mermaid diagram, and step it through with stubbed tools and a stubbed LLM. No Anvil needed — all tools are mocked.

## Setup

In [ ]:
import sys, pathlib
_ROOT = pathlib.Path.cwd().resolve()
if (_ROOT / 'shared').is_dir():
    sys.path.insert(0, str(_ROOT))
elif (_ROOT.parent / 'shared').is_dir():
    sys.path.insert(0, str(_ROOT.parent))


In [ ]:
from consumer.graph import build_graph
from shared.config import Config
import json

CONSUMER = '0x5de4111afa1a4b94908f83103eb1f1706367c2e68ca870fc3fb9a804cdab365a'
cfg = Config(consumer_private_key=CONSUMER)

## Build (stub tools and LLM)

In [ ]:
fake_catalog = [
    {'packageId': 'small',  'mbps': 2, 'durationSeconds': 600,
     'priceWei': 10**16, 'availableSlots': 1},
    {'packageId': 'medium', 'mbps': 5, 'durationSeconds': 600,
     'priceWei': 2*10**16, 'availableSlots': 1},
    {'packageId': 'large',  'mbps': 8, 'durationSeconds': 600,
     'priceWei': 8*10**16, 'availableSlots': 1},
]

async def discover(url): return json.dumps({'name': 'P', 'version': '1',
    'skills': ['get_catalog', 'request_quote', 'activate']})
async def browse(url): return json.dumps(fake_catalog)
async def quote(url, pkg): return json.dumps({
    'agreementId': '777', 'priceWei': 2*10**16,
    'bandwidthMbps': 5, 'durationSeconds': 600})
def lock(aid): return 'OK 0xdeadbeef'
def settle(aid): return 'OK tokenId=99'
async def present(url, tid): return json.dumps(
    {'status': 'active', 'bandwidthMbps': 5, 'tokenId': tid})
def verify(tid): return json.dumps({
    'ok': True, 'owner': '0xC', 'ownerIsConsumer': True,
    'agreementId': 777, 'mbps': 5, 'durationSeconds': 600,
    'secondsRemaining': 600, 'endpoint': 'clab://pe1/eth-1.100'})

tools = {'discover_provider': discover, 'browse_catalog': browse,
         'request_quote': quote, 'lock_payment': lock,
         'await_settlement': settle, 'present_credential': present,
         'verify_credential': verify}

from langchain_ollama import ChatOllama
class _R:
    def __init__(self, c): self.content = c
async def fake_ainvoke(self, prompt, *a, **kw):
    return _R('medium' if 'EXACTLY ONE WORD' in prompt else 'ok')
ChatOllama.ainvoke = fake_ainvoke

graph = build_graph(cfg, tools)
print('graph compiled')

## Inspect — the state machine

In [ ]:
print(graph.get_graph().draw_mermaid())

## Run — stream node by node

In [ ]:
initial = {'user_message': 'I need 5 Mbps',
           'provider_url': 'http://provider:8002',
           'log': [], 'thinking': []}
async for step in graph.astream(initial):
    for node, output in step.items():
        keys = list(output.keys()) if isinstance(output, dict) else type(output).__name__
        print(f'{node:18s} → {keys}')

## Final state

In [ ]:
result = await graph.ainvoke(initial)
print('final:', result['final_response'])
print('chosen tier:', result['chosen_tier'])
print('agreement:', result['agreement_id'])
print('tokenId:', result['token_id'])